In [2]:
"""
Train & compare Random Forest vs XGBoost on XAU (Gold) hourly data.
Pipeline:
  1. Load 1-min CSV → resample to 1h
  2. Build 65 features (same logic as Features.ipynb)
  3. Target = next-bar return (horizon=1)
  4. Time-series split (no shuffle)
  5. Train RF + XGBoost, evaluate MAE / RMSE / Direction Accuracy / R²
  6. Print comparison table + feature importances
"""

import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
CSV_PATH = "data/XAU_1m_data.csv"   # path to 1-min CSV (Open/High/Low/Close/Volume)
OUT_DIR = "model_output"
os.makedirs(OUT_DIR, exist_ok=True)

N_FEATURES_SELECTED = 25
TARGET_HORIZON = 1
RANDOM_STATE = 42
TEST_SIZE = 0.20          # last 20% of time as test
VAL_SIZE = 0.10           # of train portion used for early stopping (XGB)

# ---------------------------------------------------------------------------
# Feature engineering (copied & adapted from Features.ipynb)
# ---------------------------------------------------------------------------
def _rsi(close: pd.Series, period: int) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def _atr(df: pd.DataFrame, period: int) -> pd.Series:
    hl = df["High"] - df["Low"]
    hc = (df["High"] - df["Close"].shift()).abs()
    lc = (df["Low"] - df["Close"].shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    return tr.rolling(period).mean()


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """df must have Open/High/Low/Close/Volume indexed by time."""
    f = pd.DataFrame(index=df.index)
    close, high, low, vol = df["Close"], df["High"], df["Low"], df["Volume"]

    # returns & momentum
    for lag in [1, 2, 3, 5, 8, 13, 21, 34]:
        f[f"ret_{lag}"] = close.pct_change(lag)
    for lag in [1, 3, 5, 10]:
        f[f"mom_{lag}"] = close - close.shift(lag)

    # explicit lag features (assumes 1h bars)
    for lag in [1, 3, 5, 10]:
        f[f"close_lag_{lag}h"] = close.shift(lag)
        f[f"ret_lag_{lag}h"] = close.pct_change(lag)
    lag_10d = 10 * 24
    f["close_lag_10d"] = close.shift(lag_10d)
    f["ret_lag_10d"] = close.pct_change(lag_10d)

    # moving averages
    for w in [5, 10, 20, 50, 100, 200]:
        ma = close.rolling(w).mean()
        f[f"ma_{w}"] = ma
        f[f"px_over_ma_{w}"] = close / ma - 1

    # volatility
    for w in [5, 10, 20, 50]:
        f[f"vol_{w}"] = close.pct_change().rolling(w).std()
    f["atr_14"] = _atr(df, 14)
    f["atr_50"] = _atr(df, 50)
    f["hl_range"] = (high - low) / close
    f["oc_range"] = (close - df["Open"]) / df["Open"]

    # Bollinger
    bb_ma = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    f["bb_upper"] = bb_ma + 2 * bb_std
    f["bb_lower"] = bb_ma - 2 * bb_std
    f["bb_pct_b"] = (close - f["bb_lower"]) / (f["bb_upper"] - f["bb_lower"])
    f["bb_width"] = (f["bb_upper"] - f["bb_lower"]) / bb_ma

    # RSI
    for p in [7, 14, 21]:
        f[f"rsi_{p}"] = _rsi(close, p)

    # MACD
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    f["macd"] = macd
    f["macd_signal"] = macd.ewm(span=9, adjust=False).mean()
    f["macd_hist"] = f["macd"] - f["macd_signal"]

    # volume
    for w in [5, 10, 20]:
        f[f"vol_ma_{w}"] = vol.rolling(w).mean()
    f["vol_change"] = vol.pct_change()
    f["vol_zscore_20"] = (vol - vol.rolling(20).mean()) / vol.rolling(20).std()

    # candle shape
    f["body"] = (close - df["Open"]).abs() / df["Open"]
    f["upper_wick"] = (high - df[["Open", "Close"]].max(axis=1)) / df["Open"]
    f["lower_wick"] = (df[["Open", "Close"]].min(axis=1) - low) / df["Open"]

    # calendar / session
    f["hour"] = df.index.hour
    f["dayofweek"] = df.index.dayofweek
    f["is_london_session"] = df.index.hour.isin(range(7, 16)).astype(int)
    f["is_ny_session"] = df.index.hour.isin(range(12, 21)).astype(int)
    f["month"] = df.index.month

    # optional external series
    for col in df.columns:
        if col.endswith("_close"):
            f[f"{col}_ret_1"] = df[col].pct_change()
            f[f"{col}_ret_5"] = df[col].pct_change(5)

    return f


def make_target(df: pd.DataFrame, horizon: int = TARGET_HORIZON) -> pd.Series:
    return df["Close"].pct_change(horizon).shift(-horizon)


def select_top_features(X: pd.DataFrame, y: pd.Series, n: int = N_FEATURES_SELECTED) -> list:
    mask = X.notna().all(axis=1) & y.notna()
    rf = RandomForestRegressor(
        n_estimators=150, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X.loc[mask], y.loc[mask])
    importances = pd.Series(rf.feature_importances_, index=X.columns)
    top = importances.sort_values(ascending=False).head(n).index.tolist()
    return top, importances


# ---------------------------------------------------------------------------
# Metrics helpers
# ---------------------------------------------------------------------------
def direction_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.sign(y_true) == np.sign(y_pred)))


def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "DirAcc": direction_accuracy(y_true, y_pred),
    }


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    print("=" * 60)
    print("XAU Hourly – RF vs XGBoost Training & Comparison")
    print("=" * 60)

    # 1. Load & resample
    print(f"\n[1] Loading {CSV_PATH} ...")
    raw = (
        pd.read_csv(CSV_PATH, parse_dates=["Date"])
        .set_index("Date")
        .sort_index()
    )
    print(f"    1-min bars: {len(raw):,}  |  {raw.index.min()} → {raw.index.max()}")

    df_1h = (
        raw.resample("1h")
        .agg({"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"})
        .dropna()
    )
    print(f"    hourly bars: {len(df_1h):,}")

    # 2. Features + target
    print("\n[2] Building features ...")
    feats = build_features(df_1h)
    y = make_target(df_1h)
    print(f"    raw features: {feats.shape[1]}")

    # drop rows with any NaN (from rolling windows + forward target)
    data = feats.join(y.rename("target")).dropna()
    X_all = data.drop(columns=["target"])
    y_all = data["target"]
    print(f"    clean samples: {len(data):,}")

    # 3. Feature selection (RF importance on full clean set)
    print(f"\n[3] Selecting top {N_FEATURES_SELECTED} features ...")
    top_feats, all_imp = select_top_features(X_all, y_all, N_FEATURES_SELECTED)
    print("    top features:", top_feats)
    X = X_all[top_feats]

    # 4. Time-series split (no shuffle)
    n = len(X)
    test_start = int(n * (1 - TEST_SIZE))
    X_trainval, X_test = X.iloc[:test_start], X.iloc[test_start:]
    y_trainval, y_test = y_all.iloc[:test_start], y_all.iloc[test_start:]

    val_start = int(len(X_trainval) * (1 - VAL_SIZE))
    X_train, X_val = X_trainval.iloc[:val_start], X_trainval.iloc[val_start:]
    y_train, y_val = y_trainval.iloc[:val_start], y_trainval.iloc[val_start:]

    print(f"\n[4] Split sizes")
    print(f"    train : {len(X_train):,}  ({X_train.index.min()} → {X_train.index.max()})")
    print(f"    val   : {len(X_val):,}  ({X_val.index.min()} → {X_val.index.max()})")
    print(f"    test  : {len(X_test):,}  ({X_test.index.min()} → {X_test.index.max()})")

    # 5. Train models
    print("\n[5] Training models ...")

    # --- Random Forest ---
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_metrics = evaluate(y_test.values, rf_pred)
    print("    RF done")

    # --- XGBoost ---
    xgb = XGBRegressor(
        n_estimators=800,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=50,
        eval_metric="rmse",
    )
    xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    xgb_pred = xgb.predict(X_test)
    xgb_metrics = evaluate(y_test.values, xgb_pred)
    print(f"    XGB done (best iteration: {xgb.best_iteration})")

    # 6. Comparison table
    print("\n" + "=" * 60)
    print("TEST SET RESULTS (last 20% of time)")
    print("=" * 60)
    comparison = pd.DataFrame(
        {"RandomForest": rf_metrics, "XGBoost": xgb_metrics}
    ).T
    comparison = comparison[["MAE", "RMSE", "R2", "DirAcc"]]
    comparison["MAE"] = comparison["MAE"].map(lambda x: f"{x:.6f}")
    comparison["RMSE"] = comparison["RMSE"].map(lambda x: f"{x:.6f}")
    comparison["R2"] = comparison["R2"].map(lambda x: f"{x:.4f}")
    comparison["DirAcc"] = comparison["DirAcc"].map(lambda x: f"{x:.2%}")
    print(comparison.to_string())

    # decide winner (lower RMSE preferred, then higher DirAcc)
    rf_rmse = rf_metrics["RMSE"]
    xgb_rmse = xgb_metrics["RMSE"]
    if xgb_rmse < rf_rmse * 0.98:
        winner = "XGBoost"
    elif rf_rmse < xgb_rmse * 0.98:
        winner = "RandomForest"
    else:
        winner = "XGBoost" if xgb_metrics["DirAcc"] >= rf_metrics["DirAcc"] else "RandomForest"

    print(f"\n>>> Winner (by RMSE / DirAcc): {winner}")

    # 7. Feature importances
    print("\n[Feature Importances – top 10]")
    rf_imp = pd.Series(rf.feature_importances_, index=top_feats).sort_values(ascending=False)
    xgb_imp = pd.Series(xgb.feature_importances_, index=top_feats).sort_values(ascending=False)
    imp_df = pd.DataFrame({"RF": rf_imp, "XGB": xgb_imp})
    print(imp_df.head(10).to_string())

    # 8. Save artifacts
    comparison.to_csv(f"{OUT_DIR}/metrics_comparison.csv")
    imp_df.to_csv(f"{OUT_DIR}/feature_importances.csv")
    pd.Series(top_feats).to_csv(f"{OUT_DIR}/selected_features.csv", index=False, header=["feature"])

    pred_df = pd.DataFrame(
        {
            "y_true": y_test,
            "rf_pred": rf_pred,
            "xgb_pred": xgb_pred,
        },
        index=X_test.index,
    )
    pred_df.to_csv(f"{OUT_DIR}/test_predictions.csv")

    print(f"\nArtifacts saved to ./{OUT_DIR}/")
    print("Done.")


if __name__ == "__main__":
    main()

XAU Hourly – RF vs XGBoost Training & Comparison

[1] Loading data/XAU_1m_data.csv ...
    1-min bars: 729,684  |  2024-01-02 01:00:00 → 2026-02-27 05:41:00
    hourly bars: 12,230

[2] Building features ...
    raw features: 65
    clean samples: 11,881

[3] Selecting top 25 features ...
    top features: ['bb_lower', 'ma_20', 'vol_ma_20', 'ma_100', 'px_over_ma_200', 'ma_200', 'mom_10', 'ma_10', 'close_lag_10h', 'close_lag_10d', 'macd_hist', 'vol_zscore_20', 'upper_wick', 'ma_50', 'ret_2', 'px_over_ma_100', 'bb_width', 'atr_14', 'atr_50', 'px_over_ma_5', 'px_over_ma_10', 'close_lag_3h', 'ret_13', 'hl_range', 'close_lag_1h']

[4] Split sizes
    train : 8,553  (2024-01-16 13:00:00 → 2025-07-01 21:00:00)
    val   : 951  (2025-07-01 22:00:00 → 2025-08-29 07:00:00)
    test  : 2,377  (2025-08-29 08:00:00 → 2026-02-27 04:00:00)

[5] Training models ...
    RF done
    XGB done (best iteration: 8)

TEST SET RESULTS (last 20% of time)
                   MAE      RMSE       R2  DirAcc
Random